# EDA 2 Rerun — Hierarchy & Divergence - 20260625
**Phases C + D of the EDA pipeline**

How the deprivation hierarchy structures cascade and counter-cascade patterns, and where the cascade ↔ counter-cascade mirror breaks down.

### Depends on
**`msoa_cascade_features_enriched_20260625.csv`** produced by `eda_1_rerun_metric_landscape_20260625.ipynb`.

---

## 7. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
from pyprojroot import here

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150, 'savefig.bbox': 'tight'})

ROOT = here()
DATA_DIR   = ROOT / 'outputs'
OUTPUT_DIR = ROOT / 'outputs/rerun_eda_figs_20260625'  
GEO_PATH   = ROOT / 'data/london_msoa_2011.geojson'  
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from map_utils import load_london_msoa, plot_london_choropleth, plot_london_categorical

df = pd.read_csv(DATA_DIR / 'msoa_cascade_features_enriched_20260625.csv')
print(f'Loaded: {df.shape[0]} MSOAs, {df.shape[1]} columns')

CASCADE_METRICS = ['Net_Cascade', 'CFI_Churn', 'CFI_Rate', 'Pct_Inflow_Wealthier']
COUNTER_METRICS = ['Net_Counter', 'Counter_Churn', 'Counter_Rate', 'Pct_Outflow_Wealthier']

gdf = load_london_msoa(GEO_PATH, df)

---
## Phase C — How Does Wealth_Decile Structure Everything?

Before interpreting any individual MSOA, we need to understand what "normal" looks like at each position in the deprivation hierarchy.

---
## 8. Decile Profiles — All Metrics

Grouped by Wealth_Decile: mean ± standard deviation for cascade and counter-cascade metrics, side by side.

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(16, 20), sharex=True)

pairs = list(zip(CASCADE_METRICS, COUNTER_METRICS))
casc_color, counter_color = '#2166ac', '#d6604d'

for row_idx, (casc, counter) in enumerate(pairs):
    for yr_idx, yr in enumerate(['11', '21']):
        ax = axes[row_idx, yr_idx]
        
        for metric, color, label in [(casc, casc_color, f'{casc}'),
                                      (counter, counter_color, f'{counter}')]:
            col = f'{metric}_{yr}'
            grp = df.groupby('Wealth_Decile')[col].agg(['mean', 'std'])
            ax.errorbar(grp.index, grp['mean'], yerr=grp['std'],
                        fmt='o-', color=color, capsize=3, markersize=5,
                        label=label, alpha=0.8, lw=1.5)
        
        ax.set_title(f'{casc} vs {counter} (20{yr})', fontsize=10)
        ax.set_xlabel('Wealth Decile' if row_idx == 3 else '')
        ax.legend(fontsize=8)
        ax.set_xticks(range(1, 11))

fig.suptitle('Decile Profiles: Cascade vs Counter-Cascade (mean ± SD)',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_06_decile_profiles_all_metrics.png')
plt.show()

### Fig 06 Interpretation:

- The deprivation hierarchy imposes the same fundamental geometry in both census periods:
  - `Net_Cascade`, `Net_Counter`, and percentage composition metrics (`Pct_Inflow_Wealthier`, `Pct_Outflow_Wealthier`) decline monotonically from deprived to wealthy deciles
  - `CFI_Churn`, `Counter_Churn`, `CFI_Rate`, and `Counter_Rate` show an inverted-U peaking in the middle deciles
The deprivation hierarchy is the dominant structural force shaping cascade dynamics, and that structure was stable across a decade that included the COVID-affected 2021 Census.

In [ ]:
# ── 8a. Cross-year profile stability ─────────────────────────────
# Spearman ρ between the 2011 and 2021 decile-mean vectors for each metric.
# Perfect ρ = 1.0 means the rank-ordering of deciles is identical across years.

print('=== Cross-Year Profile Correlation (Spearman ρ of decile means) ===')
print(f'{"Metric pair":<45s} {"ρ":>6s}  {"p":>10s}')
print('-' * 65)

all_metrics = CASCADE_METRICS + COUNTER_METRICS
profile_rhos = {}

for m in all_metrics:
    g11 = df.groupby('Wealth_Decile')[f'{m}_11'].mean()
    g21 = df.groupby('Wealth_Decile')[f'{m}_21'].mean()
    rho, p = stats.spearmanr(g11, g21)
    profile_rhos[m] = rho
    print(f'{m + " (2011 vs 2021)":<45s} {rho:>6.4f}  {p:>10.2e}')

print(f'\nRange: ρ = {min(profile_rhos.values()):.2f} – {max(profile_rhos.values()):.2f}')

### Rerun change: 

Most rho numbers decrease slightly or keep the same as 1.

Only rho of CFI_Rate increased to 1.

In [ ]:
# ── 8b. Volume suppression: % change in decile means (2021 vs 2011) ──
# Quantifies how much each metric dropped (or rose) between census periods.

print('=== Percentage Change in Decile Means: (2021/2011 − 1) × 100 ===')
print(f'{"Metric":<25s}  ' + '  '.join([f'D{d:>2d}' for d in range(1, 11)]))
print('-' * 95)

for m in ['CFI_Churn', 'Counter_Churn', 'CFI_Rate', 'Counter_Rate']:
    g11 = df.groupby('Wealth_Decile')[f'{m}_11'].mean()
    g21 = df.groupby('Wealth_Decile')[f'{m}_21'].mean()
    pct = ((g21 / g11) - 1) * 100
    pct = pct.replace([np.inf, -np.inf], np.nan)
    vals = [f'{v:+5.1f}%' if not np.isnan(v) else ' edge' for v in pct]
    print(f'{m:<25s}  ' + '  '.join(vals))

### Rerun change:

In general, there are more increase, especially concentrated on most deprived deciles (D1-D4).

There were all decreases for all metrics in every deciles.

> **Need more interpretation here.**

In [ ]:
# ── 8c. Cascade–counter gap by decile ────────────────────────────
# Counter metric minus its cascade twin: positive = counter-cascade > cascade.
# A widening gap from 2011→2021 means the 'reverse' direction grew relative
# to the 'standard gentrification' direction.

print('=== Cascade ↔ Counter-Cascade Gap (Counter − Cascade, decile means) ===')
pairs = list(zip(CASCADE_METRICS, COUNTER_METRICS))

for casc, counter in pairs:
    print(f'\n--- {counter} − {casc} ---')
    for yr in ['11', '21']:
        gap = (df.groupby('Wealth_Decile')[f'{counter}_{yr}'].mean()
               - df.groupby('Wealth_Decile')[f'{casc}_{yr}'].mean())
        vals = '  '.join([f'D{d}:{v:+.1f}' for d, v in gap.items()])
        print(f'  20{yr}: {vals}')

### Rerun change:

The gap between cascade and counter looks mainly identical.

### Result Interpretation:


---
## 9. Derived Metrics by Decile

Cascade_Dominance and Cross_Decile_Share — how they vary across the hierarchy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cascade Dominance by decile
ax = axes[0]
for yr, color, ls in [('11', '#2166ac', '--'), ('21', '#b2182b', '-')]:
    col = f'Cascade_Dominance_{yr}'
    grp = df.groupby('Wealth_Decile')[col].agg(['mean', 'std'])
    ax.errorbar(grp.index, grp['mean'], yerr=grp['std'],
                fmt='o'+ls, color=color, capsize=3, markersize=5,
                label=f'20{yr}', alpha=0.8, lw=1.5)

ax.axhline(0.5, color='black', ls=':', lw=1, label='Midline (0.5)')
ax.set_xlabel('Wealth Decile')
ax.set_ylabel('Cascade Dominance')
ax.set_title('Cascade Dominance by Decile')
ax.legend(fontsize=9)
ax.set_xticks(range(1, 11))

# Cross Decile Share by decile
ax = axes[1]
for yr, color, ls in [('11', '#2166ac', '--'), ('21', '#b2182b', '-')]:
    col = f'Cross_Decile_Share_{yr}'
    grp = df.groupby('Wealth_Decile')[col].agg(['mean', 'std'])
    ax.errorbar(grp.index, grp['mean'], yerr=grp['std'],
                fmt='o'+ls, color=color, capsize=3, markersize=5,
                label=f'20{yr}', alpha=0.8, lw=1.5)

ax.set_xlabel('Wealth Decile')
ax.set_ylabel('Cross-Decile Share')
ax.set_title('Cross-Decile Share by Decile')
ax.legend(fontsize=9)
ax.set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_07_derived_metrics_by_decile.png')
plt.show()

### Rerun changes:

Still, 
- `Cascade_Dominance` plot keeps the same; 
- `Cross_Decile_Share` becomes more overlapping with each other. There were more clear gap between 2 lines in previous EDA resuluts.


### Result Interpretation:


---
## 10. Temporal Change (Δ) by Decile

How did cascade, counter-cascade, and derived metrics shift between 2011 and 2021 broken down by decile?

In [ ]:
delta_metrics = [f'Delta_{m}' for m in CASCADE_METRICS + COUNTER_METRICS]
delta_derived  = ['Delta_Cascade_Dominance', 'Delta_Cross_Decile_Share']
all_deltas = delta_metrics + delta_derived

n_metrics = len(all_deltas)
ncols = 2
nrows = (n_metrics + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharex=True)
axes_flat = axes.flatten()

for i, col in enumerate(all_deltas):
    ax = axes_flat[i]
    grp = df.groupby('Wealth_Decile')[col].agg(['mean', 'std'])
    
    colors = ['#b2182b' if v < 0 else '#2166ac' for v in grp['mean']]
    ax.bar(grp.index, grp['mean'], color=colors, alpha=0.7, edgecolor='white', lw=0.5)
    ax.errorbar(grp.index, grp['mean'], yerr=grp['std'],
                fmt='none', color='#333333', capsize=3, lw=0.8)
    ax.axhline(0, color='black', lw=0.5)
    ax.set_title(col.replace('Delta_', 'Δ '), fontsize=10)
    ax.set_xticks(range(1, 11))
    if i >= (nrows - 1) * ncols:
        ax.set_xlabel('Wealth Decile')

# Hide unused axes
for j in range(n_metrics, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle('Temporal Change (2021 − 2011) by Wealth Decile', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_08_temporal_delta_by_decile.png')
plt.show()

### Result Interpretation:

(Compare with before, there are more increase, especially for more deprived deciles.)

---
## Phase D — Where Does the Mirror Break?

~90% of MSOAs have concordant signs on Net_Cascade and Net_Counter. 

The ~10% divergent cases are the analytically interesting population.

---
## 11. Sign Concordance Analysis

In [ ]:
# ── Cross-tabulation ──────────────────────────────────────────
for yr in ['11', '21']:
    print(f'\n=== Sign Concordance 20{yr} ===')
    print(df[f'Sign_Concordance_{yr}'].value_counts().to_string())
    
    # By Wealth Decile
    ct = pd.crosstab(df['Wealth_Decile'], df[f'Sign_Concordance_{yr}'], margins=True)
    print(f'\nBy Wealth_Decile:')
    print(ct.to_string())

### Result Interpretation:
(The result pattern becomes more consistent between census periods.)


In [ ]:
# ── Zero subtypes ──────────────────────────────────────
for yr in ['11', '21']:
    nc  = df[f'Net_Cascade_{yr}']
    nco = df[f'Net_Counter_{yr}']
    tm  = df[f'Total_Migration_{yr}']

    # — Zero subtypes —
    zro = df[f'Sign_Concordance_{yr}'] == 'zero'
    no_data  = (zro & (tm == 0)).sum()
    balanced = (zro & (tm > 0) & (nc == 0) & (nco == 0)).sum()
    partial  = (zro & (tm > 0) & ((nc == 0) ^ (nco == 0))).sum()

    print(f'\n=== Zero Subtypes 20{yr} ===')
    print(f'  No migration data (disclosure control): {no_data}')
    print(f'  Genuinely balanced (both nets = 0):     {balanced}')
    print(f'  Partial zero (one net = 0, other ≠ 0):  {partial}')

In [ ]:
# ── Partial-zero detail (both census periods) ─────────────────
FLOW_COLS = ['Inflow_Wealthier', 'Outflow_Poorer', 'Net_Cascade',
             'Outflow_Wealthier', 'Inflow_Poorer', 'Net_Counter',
             'Total_Migration']

for yr in ['11', '21']:
    nc  = df[f'Net_Cascade_{yr}']
    nco = df[f'Net_Counter_{yr}']
    tm  = df[f'Total_Migration_{yr}']
    zro = df[f'Sign_Concordance_{yr}'] == 'zero'
    partial = zro & (tm > 0) & ((nc == 0) ^ (nco == 0))

    if partial.any():
        print(f'\n=== Partial-Zero MSOAs (detected in 20{yr}) ===')
        for _, row in df[partial].iterrows():
            msoa, borough, d = row['msoa11cd'], row['ladnm'], row['Wealth_Decile']
            print(f'\n  {msoa} ({borough}, D{d})')

            for comp_yr in ['11', '21']:
                tag = ' ◄ detected' if comp_yr == yr else ''
                iw, op = row[f'Inflow_Wealthier_{comp_yr}'], row[f'Outflow_Poorer_{comp_yr}']
                ow, ip = row[f'Outflow_Wealthier_{comp_yr}'], row[f'Inflow_Poorer_{comp_yr}']
                nc_v  = row[f'Net_Cascade_{comp_yr}']
                nco_v = row[f'Net_Counter_{comp_yr}']
                tm_v  = row[f'Total_Migration_{comp_yr}']
                sc    = row[f'Sign_Concordance_{comp_yr}']

                print(f'    20{comp_yr} ({sc}){tag}:')
                print(f'      Cascade:  IW={iw:.0f}  OP={op:.0f}  '
                      f'Net_Cascade={nc_v:+.0f}'
                      f'{"  ← balanced" if nc_v == 0 and tm_v > 0 else ""}')
                print(f'      Counter:  OW={ow:.0f}  IP={ip:.0f}  '
                      f'Net_Counter={nco_v:+.0f}'
                      f'{"  ← balanced" if nco_v == 0 and tm_v > 0 else ""}')
                print(f'      Total_Migration={tm_v:.0f}')

### Subtype of Divergent



In [ ]:
# ── Divergent subtypes ────────────────────────────────────────
for yr in ['11', '21']:
    div = df[f'Sign_Concordance_{yr}'] == 'divergent'
    nc  = df[f'Net_Cascade_{yr}']
    nco = df[f'Net_Counter_{yr}']

    cp = div & (nc > 0) & (nco < 0)   # cascade-positive
    co = div & (nc < 0) & (nco > 0)   # counter-positive

    print(f'\n=== Divergent Subtypes 20{yr} ({div.sum()} total) ===')
    print(f'  cascade-positive  (+NC / −NCo): {cp.sum()}')
    print(f'  counter-positive  (−NC / +NCo): {co.sum()}')

    # Decile × Subtype
    df[f'_subtype_{yr}'] = ''
    df.loc[cp, f'_subtype_{yr}'] = '+NC/−NCo'
    df.loc[co, f'_subtype_{yr}'] = '−NC/+NCo'

    print(f'\nDecile × Subtype:')
    print(pd.crosstab(df.loc[div, 'Wealth_Decile'],
                      df.loc[div, f'_subtype_{yr}'],
                      margins=True).to_string())

    print(f'\nTop boroughs by subtype:')
    for st, label in [('+NC/−NCo', 'cascade-positive'),
                      ('−NC/+NCo', 'counter-positive')]:
        sub = df[df[f'_subtype_{yr}'] == st]
        print(f'  {label}:')
        print(f'    {sub["ladnm"].value_counts().head(5).to_string()}')

    df.drop(columns=f'_subtype_{yr}', inplace=True)

### Divergent Subtype Interpretation:

**2011: (87 divergent, 45 cascade-positive, 42 counter-positive)**
- The split of subtype is nearly even. 
    - Neither subtype dominated London's divergent population at the start of the decade.
-  **Cascade-positive (Net receiver)**:
    - These areas simultaneously have "more Wealthier-in than Poorer-out" (positive `Net_Cascade`) and "less Wealthier-out than Poorer-in" (negative `Net_Counter`).
    - These areas gain residents in both directions, net receivers of residents across the hierarchy.
    - The **decile skews upper-middle**. D8 accounts for 11, D7-D9 together have 19 out of 45.
    - Top boroughs are mostly **outer-London** suburban boroughs.
        - Structural reason: mid-to-upper hierarchy suburbs are positioned to reveive mover from more deprived inner boroughs while their own established residents trade up to the most affluent edges of London or beyond.
- **Counter-positive (Net exporter):**
    - These areas lose residents in both directions. Downward outflows exceed wealthier inflows, and upward outflows exceed poorer inflows. These are **net exporters** of residents across the hierarchy.
    - The **decile profile skews lower**. D2 holds 9, D6 holds 9, and D8-D9 hold zero combined.
    - Top boroughs are **inner-London** boroughs, associated with well-documented gentrification during the 2010s.
        - Wealthier-out exceeds Poorer-in, while Wealthier-in remains low relative to Poorer-out.

**2021: (101 divergent, 52 cascade-positive, 49 counter-positive)**
- Both subtypes grew, and **with the same amount**.
- **The decile profiles became more polarised.**
- Cascade-positive:
    - The upper-hierarchy skew sharpened considerably.
    - D7 leads with 13 (up from 6 in 2011), D9 grew from 2 to 6, D2 dropped to zero.
        - The centre of gravity moved upward in the hierarchy.
    - Top boroughs shifted. 
        - Bromley and Enfield persisted.
        - These patterns moved to a different set of outer-London boroughs over the decade, even as the structural role itself persisted.
- **Counter-positive:**
    - The lower-hierarchy concentration intensified.
    - D2 holds 17 of 49 (up from 9 of 42), D7-D9 combined have 3.
    - Top boroughs show strong continuity.
        - Newham (12, up from 7)
        - Hackney (4, same as Tower Hamlets in 2011)
        - **Southwark (3) and Croydon (2) enter**
        - These boroughs experienced intensifying neighbourhood change through the 2010s. 
            - Wealthier-out increasingly exceeds Poorer-in, while Wealthier-in does not grow proportionally to compensate for Poorer-out.

In [ ]:
# ── Concordance transitions & persistence ─────────────────────
print('\n=== Concordance Transition Matrix (2011 → 2021) ===')
ct = pd.crosstab(df['Sign_Concordance_11'], df['Sign_Concordance_21'],
                 margins=True)
print(ct.to_string())

div_11 = df['Sign_Concordance_11'] == 'divergent'
div_21 = df['Sign_Concordance_21'] == 'divergent'
persistent = (div_11 & div_21).sum()

print(f'\n=== Divergence Persistence ===')
print(f'  Persistent (divergent both years): {persistent}')
print(f'  New in 2021:                       {(~div_11 & div_21).sum()}')
print(f'  Resolved by 2021:                  {(div_11 & ~div_21).sum()}')
print(f'  Persistence rate: {persistent}/{div_21.sum()}'
      f' = {persistent/div_21.sum():.1%}')
print(f'  (Expected by chance: ~{div_11.mean():.1%} of {div_21.sum()}'
      f' ≈ {div_11.mean()*div_21.sum():.0f})')


### Divergent Shift across the decade

In 2011, two subtypes pverlapped sustaintially in decile space.
- Both had presence in D3-D6.

In 2021, subtypes separated
- Cascade-positive consolidated in D5-D9.
- Counter-positive consolodated in D2-D6.
- The overlap narrowed.

The polarisation suggests that the cascade mechanism itself became more spatially stratified over the decade, with the divergent MSOAs sorting into more clearly defined roles in London's migration hierarchy.

In [ ]:
# ── Subtype persistence among persistently divergent MSOAs ────
div_both = df[div_11 & div_21].copy()

for yr in ['11', '21']:
    nc  = div_both[f'Net_Cascade_{yr}']
    nco = div_both[f'Net_Counter_{yr}']
    div_both[f'subtype_{yr}'] = ''
    div_both.loc[(nc > 0) & (nco < 0), f'subtype_{yr}'] = '+NC/−NCo'
    div_both.loc[(nc < 0) & (nco > 0), f'subtype_{yr}'] = '−NC/+NCo'

print(f'\n=== Subtype Transition (persistently divergent, n={len(div_both)}) ===')
print(pd.crosstab(div_both['subtype_11'], div_both['subtype_21'],
                  margins=True).to_string())
same = (div_both['subtype_11'] == div_both['subtype_21']).sum()
print(f'  Same subtype both years: {same}/{len(div_both)} ({same/len(div_both):.0%})')

# ── IMD change by concordance status ──────────────────────────
print(f'\n=== IMD Pctile Change by Concordance (2021) ===')
print(df.groupby('Sign_Concordance_21')['IMD_Pctile_Change']
        .agg(['count', 'mean', 'median', 'std'])
        .round(4).to_string())

### Result Interpretation:

- 45 of 101 (~45%) divergent MSOAs in 2021 were also divergent in 2011
- If 2011 divergence (~8.9% of total MSOAs) were independent of 2021 divergence, there would be roughly 9 of the 101 to be persistent by coincidence. 45 divergent MSOAs in 2021 is far above this assumption.
    - **For roughly half divergent MSOAs, divergence is a stable structural feature of those neighbourhoods**, rather than random noise.
- Divergent MSOAs also show a slightly more negative median IMD percentile change compared to concordant MSOAs (-0.015 vs -0.00).
    - Divergence may be associated with relative neighbourhood decline, but the difference is modest and need more statistical testing.

---

In [ ]:
# ── Create detailed concordance column for both years ─────────
for yr in ['11', '21']:
    col = f'Sign_Concordance_Detail_{yr}'
    df[col] = df[f'Sign_Concordance_{yr}']
    nc  = df[f'Net_Cascade_{yr}']
    nco = df[f'Net_Counter_{yr}']
    div = df[f'Sign_Concordance_{yr}'] == 'divergent'
    df.loc[div & (nc > 0) & (nco < 0), col] = 'cascade-positive'
    df.loc[div & (nc < 0) & (nco > 0), col] = 'counter-positive'

# ── Merge the new columns into gdf ───────────────────────────
detail_cols = ['msoa11cd', 'Sign_Concordance_Detail_11', 'Sign_Concordance_Detail_21']
gdf = gdf.drop(columns=[c for c in detail_cols if c in gdf.columns and c != 'msoa11cd'], errors='ignore')
gdf = gdf.merge(df[detail_cols], on='msoa11cd', how='left')

# ── Side-by-side concordance maps ─────────────────────────────
colours_detail = {
    'concordant':       '#d4d4d4',
    'cascade-positive': '#c4422d',
    'counter-positive': '#2d6ca2',
    'zero':             '#f5f1eb'
}

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# for idx, yr in enumerate(['11', '21']):
#     ax = axes[idx]
#     col = f'Sign_Concordance_Detail_{yr}'

#     plot_london_categorical(
#         gdf, column=col,
#         title=f'Sign Concordance: Detailed (20{yr})',
#         color_dict=colours_detail,
#         ax=ax
#     )

#     # Hatch zero MSOAs
#     zero = gdf[gdf[col] == 'zero']
#     zero.plot(ax=ax, facecolor='#f5f1eb', edgecolor='#333333',
#               linewidth=0.8, hatch='///')
for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    col = f'Sign_Concordance_Detail_{yr}'

    plot_london_categorical(
        gdf, column=col,
        title=f'Sign Concordance: Detailed (20{yr})',
        color_dict=colours_detail,
        ax=ax
    )

    # Hatch zero MSOAs
    zero = gdf[gdf[col] == 'zero']
    zero.plot(ax=ax, facecolor='#f5f1eb', edgecolor='#333333',
              linewidth=0.8, hatch='///')

    # ── NEW: Remove the automatic column name labels ──
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.axis('off') # Hides the axis box and ticks for a cleaner map

    # If the custom function automatically generated a legend on the left, remove it:
    if idx == 0 and ax.get_legend():
        ax.get_legend().remove()

legend_handles = [
    mpatches.Patch(color=hex_code, label=category.capitalize())
    for category, hex_code in colours_detail.items()
]
  
ax.legend(
    handles=legend_handles, 
    title='Concordance', 
    loc='lower right', 
    frameon=False,
    title_fontsize='10',
    fontsize='9'
)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_09a_sign_concordance_map.png', dpi=300, bbox_inches='tight')
plt.show()

### Fig 09a 2011vs2021 Interpretation

In [ ]:
# ── Diagnostic map: divergent MSOAs ───────────────────────────

# Add a column for the four-category version
df['Sign_Concordance_Detail_21'] = df['Sign_Concordance_21']
nc  = df['Net_Cascade_21']
nco = df['Net_Counter_21']
div = df['Sign_Concordance_21'] == 'divergent'
df.loc[div & (nc > 0) & (nco < 0), 'Sign_Concordance_Detail_21'] = 'cascade-positive'
df.loc[div & (nc < 0) & (nco > 0), 'Sign_Concordance_Detail_21'] = 'counter-positive'

colours_detail = {
    'concordant':       '#d4d4d4',
    'cascade-positive': '#c4422d',
    'counter-positive': '#2d6ca2',
    'zero':             '#f5f1eb'
}

gdf = load_london_msoa(GEO_PATH, df)

fig, ax = plot_london_categorical(
    gdf, 
    column='Sign_Concordance_Detail_21',
    title='Sign Concordance: Net_Cascade vs Net_Counter (2021)',
    color_dict=colours_detail
)

legend_handles = [
    mpatches.Patch(color=hex_code, label=category.capitalize())
    for category, hex_code in colours_detail.items()
]

ax.legend(
    handles=legend_handles, 
    title='Concordance', 
    loc='lower right', 
    frameon=False,
    title_fontsize='10',
    fontsize='9'
)


zero_msoas = gdf[gdf['Sign_Concordance_Detail_21'] == 'zero']
zero_msoas.plot(ax=ax, facecolor='#f5f1eb', edgecolor='#333333',
                linewidth=0.8, hatch='///')

plt.savefig(OUTPUT_DIR / 'fig_09b_sign_concordance_map_2021.png', dpi=300, bbox_inches='tight')
plt.show()

##### Fig 09b 2021 Interpretation
- **Zero MSOAs:**
    - These areas are **scattered** without obvious pattern, consistent with the earlier finding that 20 of 21 are disclosure artefacts rather than substantive cases.

- **Counter-positive cluster in inner-London (Net exporter)**
    - A visible concentration running through the inner-east and central boroughs
    - In the results above, these areas are predominantly lower-decile areas (D2-D5). 
    - Both outflow directions exceed their corresponding inflow directions.
        - The area is a net exporter in both directions of the hierarchy.
    - Such **inner-London concentration** are the boroughs most exposed to gentrification pressures during 2010s, where outflow to wealthier destinations operate more strongly than inflows from wealthier origins.
- **Cascade-positive ring the outer edges (Net absorber)**
    - Red polygons are disproportioantely on London's periphery.
    - These areas are the mid-to-upper hierarchy suburbs (D5-D9). 
    - These areas sit at the interface between inner-London restructuring and the suburban fringe.
        - They receive inflows from both directions of the hierarchy more than they export, acting as net absorbers rather than exporters.

Blue inside, Red outside, with the grey concordant majority filling the space between. 
- **Divergent MSOAs are not randomly scattered** (measurement noise), nor are they tightly clustered in one spot (single local anomaly). 
- **There is a clear structural spatial logic that mirrors London's deprivation gradient**
    - **Net exporters(counter-positive) operate in the inner gentrifying zone**
    - **Net receivers (cascade-positive) operate in the outer suburban ring**
    - counter-positive and cascade-positive rarely overlap geographically.



---
## 12. Cascade_Dominance Scatter

Which MSOAs sit furthest from the 0.50 midline?

In [ ]:
# ── Shared axis limits ────────────────────────────────────────
x_min = min(df['Cascade_Dominance_11'].min(),
            df['Cascade_Dominance_21'].min())
x_max = max(df['Cascade_Dominance_11'].max(),
            df['Cascade_Dominance_21'].max())
y_min = 0
y_max = max(df['CFI_Churn_11'].max(),
            df['CFI_Churn_21'].max()) * 1.05
 
# ── Discrete colourmap ────────────────────────────────────────
cmap = matplotlib.colormaps.get_cmap('YlOrBr_r').resampled(10)
 
# ── Figure ────────────────────────────────────────────────────
fig, axes = plt.subplots(
    1, 2, figsize=(14, 5.8),
    sharex=True, sharey=True, layout='constrained'
)
 
for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    dom_col = f'Cascade_Dominance_{yr}'
    churn_col = f'CFI_Churn_{yr}'
 
    # Zone shading
    ax.axvspan(x_min - 0.01, 0.5,
               alpha=0.035, color='#4575b4', zorder=0)
    ax.axvspan(0.5, x_max + 0.01,
               alpha=0.035, color='#d73027', zorder=0)
 
    # Main scatter
    scatter = ax.scatter(
        df[dom_col], df[churn_col],
        c=df['Wealth_Decile'], cmap=cmap, vmin=1, vmax=10,
        s=16, alpha=0.55, edgecolor='none', zorder=3
    )
 
    # LOWESS trend
    sns.regplot(
        x=df[dom_col], y=df[churn_col],
        lowess=True, scatter=False, ax=ax,
        color='#333333',
        line_kws={'lw': 2, 'alpha': 0.85, 'zorder': 6}
    )
 
    # Divergent ring markers
    div_mask = df[f'Sign_Concordance_{yr}'] == 'divergent'
    ax.scatter(
        df.loc[div_mask, dom_col],
        df.loc[div_mask, churn_col],
        facecolors='none', edgecolors='#b2182b',
        s=48, lw=0.9, zorder=8,
        label=f'Divergent (n = {div_mask.sum()})'
    )
 
    # Midline
    ax.axvline(0.5, color='#222222', ls='--', lw=1,
               alpha=0.7, zorder=5)
 
    # Pearson r annotation
    r, p = stats.pearsonr(df[dom_col], df[churn_col])
    sig = ('***' if p < 0.001 else
           '**'  if p < 0.01  else
           '*'   if p < 0.05  else '')
    ax.text(
        0.97, 0.96, f'r = {r:.2f}{sig}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(facecolor='white', alpha=0.85,
                  edgecolor='#cccccc', boxstyle='round,pad=0.3')
    )
 
    # Zone labels
    ax.text(0.38, y_max * 0.96, 'Counter-cascade\ndominated',
            ha='center', va='top', fontsize=7.5,
            color='#4575b4', alpha=0.6)
    ax.text(0.62, y_max * 0.96, 'Cascade\ndominated',
            ha='center', va='top', fontsize=7.5,
            color='#d73027', alpha=0.6)
 
    # Axes
    ax.set_title(f'20{yr}', fontsize=12, fontweight='bold', pad=8)
    ax.set_xlabel('Cascade Dominance', fontsize=10)
    ax.set_ylabel('CFI Churn' if idx == 0 else '', fontsize=10)
    ax.set_xlim(x_min - 0.005, x_max + 0.005)
    ax.set_ylim(y_min, y_max)
    ax.legend(loc='upper left', fontsize=8, frameon=True,
              framealpha=0.9, edgecolor='#cccccc')
    ax.tick_params(labelsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
 
# ── Colourbar ─────────────────────────────────────────────────
cbar = fig.colorbar(
    scatter, ax=axes, shrink=0.78, pad=0.02, aspect=25
)
cbar.set_label('Wealth Decile (1 = most deprived)', fontsize=9.5)
cbar.set_ticks(range(1, 11))
cbar.ax.tick_params(labelsize=8)
 
# ── Title ─────────────────────────────────────────────────────
fig.suptitle(
    'Cascade Dominance vs CFI Churn by Wealth Decile',
    fontsize=13, fontweight='bold'
)
 
plt.savefig(
    OUTPUT_DIR / 'fig_10_dominance_scatter_refined.png',
    dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
# ── §12b  Cascade Dominance Scatter — Companion Diagnostics ───

print('=' * 65)
print('§12 Companion Diagnostics: Cascade Dominance vs CFI Churn')
print('=' * 65)

# ── 1. Midline split ──────────────────────────────────────────
print('\n── 1. Share of MSOAs below / above 0.50 midline ──')
for yr in ['11', '21']:
    dom = df[f'Cascade_Dominance_{yr}']
    below = (dom < 0.5).mean() * 100
    above = (dom > 0.5).mean() * 100
    equal = (dom == 0.5).mean() * 100
    print(f'  20{yr}: {below:.1f}% below  |  {above:.1f}% above  |  '
          f'{equal:.1f}% at 0.50')

# ── 2. Temporal shift in dominance ────────────────────────────
print('\n── 2. Temporal shift in Cascade Dominance ──')
mean_11 = df['Cascade_Dominance_11'].mean()
mean_21 = df['Cascade_Dominance_21'].mean()
t_stat, p_val = stats.ttest_rel(
    df['Cascade_Dominance_21'],
    df['Cascade_Dominance_11']
)
print(f'  Mean 2011: {mean_11:.4f}')
print(f'  Mean 2021: {mean_21:.4f}')
print(f'  Δ mean:    {mean_21 - mean_11:+.4f}')
print(f'  Paired t-test: t = {t_stat:.2f}, p = {p_val:.2e}')

# ── 3. Correlation: Dominance vs Churn ────────────────────────
print('\n── 3. Pearson r (Cascade Dominance vs CFI Churn) ──')
for yr in ['11', '21']:
    r, p = stats.pearsonr(
        df[f'Cascade_Dominance_{yr}'],
        df[f'CFI_Churn_{yr}']
    )
    print(f'  20{yr}: r = {r:.4f}, p = {p:.2e}')

r_11, _ = stats.pearsonr(df['Cascade_Dominance_11'], df['CFI_Churn_11'])
r_21, _ = stats.pearsonr(df['Cascade_Dominance_21'], df['CFI_Churn_21'])
print(f'  Δr = {r_21 - r_11:+.4f}')

# ── 4. Divergent MSOAs: midline proximity & decile range ──────
print('\n── 4. Divergent MSOAs: positioning ──')
for yr in ['11', '21']:
    div = df[df[f'Sign_Concordance_{yr}'] == 'divergent']
    dom_col = f'Cascade_Dominance_{yr}'
    n = len(div)

    near_midline = ((div[dom_col] >= 0.45) & (div[dom_col] <= 0.55)).sum()
    deciles_present = sorted(int(d) for d in div['Wealth_Decile'].unique())

    print(f'  20{yr} (n = {n}):')
    print(f'    Within 0.45–0.55 band: {near_midline}/{n} '
          f'({near_midline / n * 100:.0f}%)')
    print(f'    Deciles present: {deciles_present}')
    print(f'    Decile distribution:')
    print(f'      {div["Wealth_Decile"].value_counts().sort_index().to_dict()}')

# ── 5. High-churn tail (top 10%): decile composition ─────────
print('\n── 5. High-churn tail (top 10%) ──')
for yr in ['11', '21']:
    churn_col = f'CFI_Churn_{yr}'
    dom_col = f'Cascade_Dominance_{yr}'
    threshold = df[churn_col].quantile(0.90)
    top = df[df[churn_col] >= threshold]

    print(f'  20{yr}: churn ≥ {threshold:.0f} (n = {len(top)})')
    print(f'    Mean dominance: {top[dom_col].mean():.4f}')
    print(f'    Decile counts: '
          f'{top["Wealth_Decile"].value_counts().sort_index().to_dict()}')

### Fig 10 Interpretation:

This plot can test among MSOAs where cascade flow or counter-cascade flow dominate, how intensely is it happening? -- **intensity of cascade-direction flows** specifically.

Cascade mechanism is real but not universal. It operates most intensely in the deprived-to-middle band. This is a minority dynamic overall, and became more diffuse between 2011 and 2021.

- **The majority of London MSOAs are counter-cascade dominated, and this intensified by 2021.**
    - The dot cloud sits relative to the 0.50 midline. 
        - In 2011, 67.6% of MSOAs already fell below 0.5; 
        - By 2021, it rose to 80.9%.
    - The mean shifted leftward, from 0.48 to 0.46, paired t = -10.17, p<0.01
        - Across London, counter-cascade flows marginally outweigh the cascade direction in most neighbourhoods.
        - **The "displacement cascade" is not the dominant flow type in the majority of MSOAs, rather it is a minority pattern concentrated in specific areas.**

- **Dominance and churn are positively coupled, but the relationship weakened between census periods**
  **Positive correlation means MSOAs with higher Cascade_Dominance also tend to have higher absolute churn.**
  **This is not just that the balance tilts cascade-ward, but the more people are actually moving across deprivation lines in the cascade direction.**
    - r = 0.53 in 2011, r = 0.46 in 2021, both p<0.01
    - MSOAs with cascade flows outweigh counter-cascade flows tend to have higher total cascade-relevant volume.
    - Cascade process is self-reinforcing. Where it dominates directionally, it also runs at higher intensity.
    - The weakening to r = 0.46 by 2021 is possibly because overall migration volumes fell but the directional balance shifted unevenly, due to COVID.

- The decile colour gradient runs counter to a naive gentrification expectation.
    - **Most deprived and mid-hierarchy deciles (D1-D6) show the highest churn and the highest dominance.**
    - **The wealthiest decile (D10) clusters in the low churn.**
        - This is partly mechanical, since D10 can't receive cascade inflows, suppressing both their dominance numerator and churn
        - It also reflects the **heaviest cascade activity concentrates in the deprived-to-middle belt of the hierarchy**, which is where gentrification thepry predicts the "rent gap" dynamics operate.
    - The high-churn tail (top 10%) is dominated by D2-D6, not the extremes.

- **"Divergent" MSOAs cluster**

---
## 13. Cascade Dominance — Diagnostic Map

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    col = f'Cascade_Dominance_{yr}'
    plot_london_choropleth(
        gdf, column=col,
        title=f'Cascade Dominance (20{yr})',
        cmap='RdBu_r', vcenter=0.5, vmin=0.25, vmax=0.75,
        legend_label='Dominance (>0.5 = cascade-led)',
        ax=ax
    )

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_11_dominance_map.png')
plt.show()

### Fig 11 Interpretation

(In the middle, there is an area with extremely red in 2011 but turned moderate warm by 2021.)

The dominance map (fig11) and the concordance map (fig 9) capture different dimensions of cascade behaviour. 

Dominance measures the volume share of cascade-directed flows; concordance measures whether cascade and counter-cascade net flows are directionally consistent. 

Since the volume balance of Divergent MSOAs is similar to concordant areas, these areas are not distinguishable on the dominance map. This confirms that sign concordance captures a structural property (directional inconsistency) that volume-based metrics miss. 

The zero MSOAs are the exception, where disclosure-suppressed areas appear as exactly 0.5 on the dominance map, providing a visual cross-check for identifying data artefacts.

In [ ]:
# ── Dominance × Concordance overlay (2011 & 2021) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for idx, yr in enumerate(['11', '21']):
    ax = axes[idx]
    col_detail = f'Sign_Concordance_Detail_{yr}'

    # Layer 1: dominance choropleth
    plot_london_choropleth(
        gdf, column=f'Cascade_Dominance_{yr}',
        title=f'Cascade Dominance with Divergent Boundaries (20{yr})',
        cmap='RdBu_r', vcenter=0.5, vmin=0.25, vmax=0.75,
        legend_label='Dominance (>0.5 = cascade-led)',
        ax=ax
    )

    # Layer 2: divergent boundaries
    cascade_pos = gdf[gdf[col_detail] == 'cascade-positive']
    counter_pos = gdf[gdf[col_detail] == 'counter-positive']
    zero_msoas  = gdf[gdf[col_detail] == 'zero']

    cascade_pos.boundary.plot(ax=ax, edgecolor='#c4422d', linewidth=1.8)
    counter_pos.boundary.plot(ax=ax, edgecolor='#2d6ca2', linewidth=1.8)
    zero_msoas.plot(ax=ax, facecolor='none', edgecolor='#333333',
                    linewidth=1.2, hatch='///')

# Shared legend (only once, outside the loop)
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], color='#c4422d', linewidth=1.8, label='Cascade-positive'),
    Line2D([0], [0], color='#2d6ca2', linewidth=1.8, label='Counter-positive'),
    Patch(facecolor='none', edgecolor='#333333', hatch='///', label='Zero'),
]
axes[1].legend(handles=legend_handles, loc='lower right',
               frameon=False, fontsize=9, title='Divergent boundaries')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_11a_dominance_concordance_overlay.png',
            dpi=300, bbox_inches='tight')
plt.show()

### Fig 11a Interpretation - 2011vs2021 Overlay

- "Divergent" MSOA boundaries (red and blue outlines) sit on top of a wide range of Dominance fills, without any consistent pattern.
    - Cascade-positive outlines are not preferntially on red-filled (high-dominance) areas.
    - Counter-positive outlines are not preferentially on blue-filled (low-dominance) areas.
    - This confirms visually that "Divergent" MSOAs have similar dominance values to their concordance neighbours.

- Dominance measures *volume balance* (how much of the total cross-decile churn is cascade vs. counter-cascade). Concordance measures *directional consistency* (whether the net signs agree).
    - An MSOA can have near-euqal cascade and counter-cascade *volumes* yet still be divergent, since the *net directions* contradict.
    - The two maps capture genuinely independent dimensions of cascade behaviour. The overlay demonstrates one can't be read from the other.

- Hatches areas on the overlay sit at exactly 0.5 dominance (white fill), confirming they are disclosure artefacts with no real flow data, rather than genuinely balanced areas.

- Temporal shift in dominance:
    - **2021 panel is noticeably bluer overall than 2011.**
        - Counter-cascade volumes grew relative to cascade volumes across London.
    - The volume shift did not change the concordance pattern
        - Inner/Outer subtype geography persisted even as the overall volume balance tilted.


In [ ]:
# ── Dominance × Concordance overlay (2021) ────────────────────
fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Layer 1: dominance choropleth as the base
plot_london_choropleth(
    gdf, column='Cascade_Dominance_21',
    title='Cascade Dominance with Divergent MSOA Boundaries (2021)',
    cmap='RdBu_r', vcenter=0.5, vmin=0.25, vmax=0.75,
    legend_label='Dominance (>0.5 = cascade-led)',
    ax=ax
)

# Layer 2: divergent MSOA boundaries on top
cascade_pos  = gdf[gdf['Sign_Concordance_Detail_21'] == 'cascade-positive']
counter_pos  = gdf[gdf['Sign_Concordance_Detail_21'] == 'counter-positive']
zero_msoas   = gdf[gdf['Sign_Concordance_Detail_21'] == 'zero']

cascade_pos.boundary.plot(ax=ax, edgecolor='#c4422d', linewidth=1.8, label='Cascade-positive')
counter_pos.boundary.plot(ax=ax, edgecolor='#2d6ca2', linewidth=1.8, label='Counter-positive')
zero_msoas.plot(ax=ax, facecolor='none', edgecolor='#333333', linewidth=1.2,
                hatch='///', label='Zero (disclosure / balanced)')

ax.legend(loc='lower right', frameon=False, fontsize=9, title='Divergent boundaries')

plt.savefig(OUTPUT_DIR / 'fig_11b_dominance_concordance_overlay_2021.png', dpi=300, bbox_inches='tight')
plt.show()

---
*Next: `eda_3_typology_validation.ipynb` (Phases E + F)*

### Next Step:

Refine ineterpretation of section 8, fig 7 cds; fig 8; fig 9a; fig 10 black lines + divergent MSOAs in 2021; fig 11.